# Evaluation & Handoff

This notebook loads processed data and saved models, evaluates their performance on held-out test data (or a chronological split for forecasting), and writes a short evaluation summary suitable for handoff to operations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

BASE = Path('..')
PROCESSED = BASE / 'data' / 'processed_medical_appointments.csv'
MODELS = BASE / 'models'
OUT = BASE / 'release'
OUT.mkdir(parents=True, exist_ok=True)

print('Looking for processed data at', PROCESSED)
if not PROCESSED.exists():
    print('Processed data not found. Please run preprocessing notebook first.')
else:
    df = pd.read_csv(PROCESSED, low_memory=False, parse_dates=[c for c in ['appointment_date','appointment_date_continuous'] if c in pd.read_csv(PROCESSED,nrows=0).columns])
    print('Loaded processed data:', df.shape)

## No-show model evaluation

In [ ]:
metrics = {}
no_show_model_path = MODELS / 'no_show_best.joblib'
if not no_show_model_path.exists():
    print('No-show model not found at', no_show_model_path)
else:
    pipe = joblib.load(no_show_model_path)
    # Prepare X,y similar to training script
    target = 'no_show'
    if target not in df.columns:
        print('Target column no_show missing in processed data.')
    else:
        y = df[target].astype(int)
        drop_cols = [c for c in ['appointment_date','appointment_time','appointment_date_continuous'] if c in df.columns]
        X = df.drop(columns=[target] + drop_cols, errors='ignore')
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
        y_pred = pipe.predict(X_test)
        try:
            y_proba = pipe.predict_proba(X_test)[:,1]
        except Exception:
            y_proba = None
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred).tolist()
        metrics['no_show'] = {'f1': f1, 'roc_auc': roc, 'precision': prec, 'recall': rec, 'confusion_matrix': cm}
        print('No-show evaluation:')
        print(json.dumps(metrics['no_show'], indent=2))
        # save a small sample predictions file
        sample_out = OUT / 'no_show_sample_predictions.csv'
        X_test_copy = X_test.copy()
        X_test_copy['y_true'] = y_test
        X_test_copy['y_pred'] = y_pred
        if y_proba is not None:
            X_test_copy['y_proba'] = y_proba
        X_test_copy.head(200).to_csv(sample_out, index=False)
        print('Saved sample predictions to', sample_out)

## Demand forecasting evaluation

In [ ]:
# aggregate daily counts
date_col = 'appointment_date' if 'appointment_date' in df.columns else 'appointment_date_continuous'
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
daily = df.groupby(df[date_col].dt.date).size().reset_index(name='count')
daily['date'] = pd.to_datetime(daily[date_col]) if date_col in daily.columns else pd.to_datetime(daily['index'])
daily = daily[['date','count']].set_index('date').sort_index()
print('Daily series length:', len(daily))

def create_lag_features(series, lags=[1,7,14], rolling_windows=[7,14]):
    df_l = pd.DataFrame({'y': series})
    for l in lags:
        df_l[f'lag_{l}'] = df_l['y'].shift(l)
    for w in rolling_windows:
        df_l[f'roll_mean_{w}'] = df_l['y'].shift(1).rolling(w).mean()
        df_l[f'roll_std_{w}'] = df_l['y'].shift(1).rolling(w).std()
    df_l = df_l.dropna()
    return df_l

metrics['demand'] = {}
# LightGBM evaluation
lgb_path = MODELS / 'demand_lgbm.pkl'
if lgb_path.exists():
    try:
        lgbm = joblib.load(lgb_path)
        features = create_lag_features(daily['count'])
        X = features.drop(columns=['y'])
        y = features['y']
        # chronological split: last 30 days as test if available
        test_days = min(30, len(X)//5)
        split_idx = len(X) - test_days
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
        preds = lgbm.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        rmse = mean_squared_error(y_test, preds, squared=False)
        mape = np.mean(np.abs((y_test - preds) / np.clip(y_test, 1e-8, None))) * 100
        r2 = r2_score(y_test, preds)
        metrics['demand']['lightgbm'] = {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}
        print('LightGBM demand metrics:', metrics['demand']['lightgbm'])
    except Exception as e:
        print('LightGBM evaluation failed:', e)
else:
    print('No LightGBM model found at', lgb_path)

# SARIMAX evaluation if exists
sarimax_path = MODELS / 'demand_sarimax.pkl'
if sarimax_path.exists():
    try:
        import pickle
        with open(sarimax_path,'rb') as f:
            sarimax_res = pickle.load(f)
        # the saved results object can forecast; use last 30 days
        test_days = min(30, len(daily))
        test = daily['count'].iloc[-test_days:]
        preds = sarimax_res.forecast(steps=len(test))
        mae = mean_absolute_error(test, preds)
        rmse = mean_squared_error(test, preds, squared=False)
        mape = np.mean(np.abs((test - preds) / np.clip(test, 1e-8, None))) * 100
        r2 = r2_score(test, preds)
        metrics['demand']['sarimax'] = {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}
        print('SARIMAX metrics:', metrics['demand']['sarimax'])
    except Exception as e:
        print('SARIMAX evaluation failed:', e)
else:
    print('No SARIMAX model found at', sarimax_path)

# Save metrics to release folder
with open(OUT / 'evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=lambda x: None)
print('Saved evaluation metrics to', OUT / 'evaluation_metrics.json')

## Handoff checklist

- [ ] Confirm `models/` contains `no_show_best.joblib` and `demand_lgbm.pkl` (or other preferred models).
- [ ] Run `streamlit run app.py` locally to validate the dashboard and UX.
- [ ] Provide scheduled retraining cadence and responsible owner.
- [ ] Add operational monitoring: prediction latency, model drift, and accuracy tracking.

Copy of this notebook and `release/evaluation_metrics.json` should be provided to stakeholders along with `docs/MODEL_CARD.md` and `docs/DATA_DICTIONARY.md`.